# G2 Correlation Analysis Workflow

This notebook demonstrates the complete G2 correlation analysis workflow:
from loading data to fitting, visualization, and extracting physical parameters.

## Prerequisites

- xpcsviewer installed
- An XPCS Multitau result HDF5 file
- matplotlib for visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
from xpcsviewer.xpcs_file import XpcsFile

DATA_FILE = "/path/to/A001_Multitau_result.hdf"
xf = XpcsFile(DATA_FILE)

print(f"Analysis type: {xf.atype}")
print(f"Label: {xf.label}")

## 2. Overview: All Q Bins

In [ ]:
q_values, t_el, g2, g2_err, labels = xf.get_g2_data()

print(f"G2 data: {g2.shape[1]} Q bins, {g2.shape[0]} delay points")
print(f"Delay time range: [{t_el.min():.2e}, {t_el.max():.2e}] s")
print(f"Q range: [{min(q_values):.4f}, {max(q_values):.4f}]")

In [ ]:
# Waterfall plot of all G2 curves
fig, ax = plt.subplots(figsize=(12, 8))

n_q = g2.shape[1]
colors = plt.cm.viridis(np.linspace(0, 1, n_q))

for i in range(n_q):
    ax.semilogx(t_el, g2[:, i], 'o-', color=colors[i], markersize=2, linewidth=0.5)

ax.set_xlabel('Delay time (s)')
ax.set_ylabel(r'$g_2(\tau)$')
ax.set_title(f'G2 Overview: {xf.label} ({n_q} Q bins)')

# Add colorbar for Q values
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(min(q_values), max(q_values)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Q (nm$^{-1}$)')

plt.tight_layout()
plt.show()

## 3. Single Q-Bin Analysis

Examine a single Q bin in detail.

In [ ]:
# Select a Q bin (0-indexed)
q_idx = 5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# G2 curve
axes[0].errorbar(t_el, g2[:, q_idx], yerr=g2_err[:, q_idx],
                 fmt='o', markersize=4, capsize=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('Delay time (s)')
axes[0].set_ylabel(r'$g_2(\tau)$')
axes[0].set_title(f'G2: {labels[q_idx]}')

# Error distribution
axes[1].hist(g2_err[:, q_idx], bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('G2 Error')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Error Distribution: {labels[q_idx]}')

plt.tight_layout()
plt.show()

## 4. G2 Fitting with XpcsFile

Use the built-in `fit_g2()` method for single-exponential fitting.

In [ ]:
# Define bounds: [baseline, tau, contrast, stretching_exp]
bounds = [
    [0.9, 1e-6, 0.0, 0.5],   # lower
    [1.1, 1e3,  0.5, 1.5],   # upper
]

fit_summary = xf.fit_g2(
    q_range=None,          # All Q bins
    t_range=None,          # All delay times
    bounds=bounds,
    fit_func='single',
)

print(f"Fit function: {fit_summary['fit_func']}")
print(f"Number of Q bins fitted: {len(fit_summary['q_val'])}")
print(f"Fit parameters shape: {fit_summary['fit_val'].shape}")
print(f"Fit line shape: {fit_summary['fit_line'].shape}")

In [ ]:
# Plot data with fits
n_plot = min(6, len(fit_summary['q_val']))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(n_plot):
    ax = axes[i]
    # Data points
    ax.errorbar(t_el, g2[:, i], yerr=g2_err[:, i],
                fmt='o', markersize=3, capsize=1, label='Data')
    # Fit line
    ax.plot(fit_summary['fit_x'], fit_summary['fit_line'][:, i],
            'r-', linewidth=2, label='Fit')
    ax.set_xscale('log')
    ax.set_xlabel('Delay time (s)')
    ax.set_ylabel(r'$g_2(\tau)$')
    ax.set_title(f'{labels[i]}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle(f'G2 Fits: {xf.label}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Extract Fit Parameters

The `fit_val` array contains the fitted parameters for each Q bin.

In [ ]:
# For single exponential: fit_val columns are
# [baseline, tau, contrast, stretching_exponent]
q_arr = np.array(fit_summary['q_val'])
fit_vals = fit_summary['fit_val']

tau_fitted = fit_vals[:, 1]      # Relaxation time
baseline_fitted = fit_vals[:, 0] # Baseline
contrast_fitted = fit_vals[:, 2] # Contrast

print("Fitted parameters per Q bin:")
print(f"{'Q':>10} {'tau':>12} {'baseline':>10} {'contrast':>10}")
print("-" * 45)
for i in range(len(q_arr)):
    print(f"{q_arr[i]:10.4f} {tau_fitted[i]:12.4e} {baseline_fitted[i]:10.4f} {contrast_fitted[i]:10.4f}")

## 6. Tau vs Q Analysis

For diffusive dynamics: $1/\tau = D \cdot q^2$

In [ ]:
# Filter out non-physical fits
valid = (tau_fitted > 0) & np.isfinite(tau_fitted)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# tau vs q
axes[0].semilogy(q_arr[valid], tau_fitted[valid], 'bo-')
axes[0].set_xlabel(r'$q$ (nm$^{-1}$)')
axes[0].set_ylabel(r'$\tau$ (s)')
axes[0].set_title(r'$\tau$ vs $q$')

# 1/tau vs q^2
axes[1].plot(q_arr[valid]**2, 1.0/tau_fitted[valid], 'ro-')
axes[1].set_xlabel(r'$q^2$ (nm$^{-2}$)')
axes[1].set_ylabel(r'$1/\tau$ (s$^{-1}$)')
axes[1].set_title(r'$1/\tau$ vs $q^2$')

# Linear fit for diffusion coefficient
if valid.sum() > 2:
    coeffs = np.polyfit(q_arr[valid]**2, 1.0/tau_fitted[valid], 1)
    D = coeffs[0]
    q2_fine = np.linspace(0, q_arr[valid].max()**2 * 1.1, 100)
    axes[1].plot(q2_fine, np.polyval(coeffs, q2_fine), 'k--',
                 label=f'D = {D:.2e} nm$^2$/s')
    axes[1].legend()

# Contrast vs q
axes[2].plot(q_arr[valid], contrast_fitted[valid], 'gs-')
axes[2].set_xlabel(r'$q$ (nm$^{-1}$)')
axes[2].set_ylabel('Contrast')
axes[2].set_title('Contrast vs $q$')

plt.tight_layout()
plt.show()

## 7. SAXS 1D Profile

The SAXS profile provides structural information about the sample.

In [ ]:
q_saxs, Iq, xlabel, ylabel = xf.get_saxs1d_data()

fig, ax = plt.subplots(figsize=(8, 5))
if Iq.ndim == 2:
    ax.loglog(q_saxs, Iq[0], '-', linewidth=1)
else:
    ax.loglog(q_saxs, Iq, '-', linewidth=1)

ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_title('SAXS 1D Profile')
plt.tight_layout()
plt.show()

## 8. Heatmap Visualization

Visualize G2 as a 2D heatmap (delay time vs Q).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# G2 heatmap: rows=delay, columns=Q
im = ax.pcolormesh(
    np.arange(g2.shape[1]),
    t_el,
    g2,
    shading='auto',
    cmap='viridis',
)
ax.set_yscale('log')
ax.set_xlabel('Q bin index')
ax.set_ylabel('Delay time (s)')
ax.set_title(f'G2 Heatmap: {xf.label}')
plt.colorbar(im, ax=ax, label=r'$g_2(\tau)$')
plt.tight_layout()
plt.show()

## 9. Cleanup

In [ ]:
xf.close()
print("Analysis complete.")

## Summary

This notebook covered:

1. Loading XPCS data with `XpcsFile`
2. Visualizing G2 curves across Q bins
3. Single-exponential fitting with `fit_g2()`
4. Extracting tau(q) for diffusion analysis
5. SAXS 1D profile visualization
6. G2 heatmap representation

Next: [03_fitting_workflow.ipynb](03_fitting_workflow.ipynb) for advanced fitting with NLSQ and Bayesian methods.